In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import torch
import random
import time

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
# trl==0.11
from trl import AutoModelForCausalLMWithValueHead, AutoModelForSeq2SeqLMWithValueHead, create_reference_model, PPOTrainer, PPOConfig
from tqdm import tqdm

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"[INFO] Using {device} device")

[INFO] Using cuda device


In [3]:
np.random.seed(88)
tqdm.pandas()

In [4]:
# 1. Sentimental Analysis (for reward system)

In [5]:
sentiment_analysis = pipeline('text-classification', 'cardiffnlp/twitter-roberta-base-sentiment-latest')

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [6]:
# Test
result_test = sentiment_analysis("nice job", function_to_apply='none', top_k=None)
result_test

[{'label': 'positive', 'score': 1.9797115325927734},
 {'label': 'neutral', 'score': -0.43869760632514954},
 {'label': 'negative', 'score': -1.8894321918487549}]

In [7]:
# 2. Linguistic Acceptability (for reward system)

In [8]:
cola_tokenizer = AutoTokenizer.from_pretrained("textattack/roberta-base-CoLA")
cola_model = AutoModelForSequenceClassification.from_pretrained("textattack/roberta-base-CoLA")
linguistic_acceptability = pipeline('text-classification', model=cola_model, tokenizer=cola_tokenizer)

Some weights of the model checkpoint at textattack/roberta-base-CoLA were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [9]:
# Test 
result_test = linguistic_acceptability("Her went to a school", function_to_apply='none', top_k=None)
result_test

[{'label': 'LABEL_0', 'score': 1.7046984434127808},
 {'label': 'LABEL_1', 'score': -1.2301501035690308}]

In [10]:
# Wrappers

In [11]:
def neutral_scores(texts):
    scores = []
    results = sentiment_analysis(texts, function_to_apply='none', top_k=None)
    for result in results:
        for label in result:
            if label['label'] == 'neutral':
                scores.append(label['score'])
    return scores
      
neutral_scores(['nice job', 'what a waste', 'hello world!', 'nothing special'])

[-0.43869760632514954,
 -0.01781720295548439,
 0.11776556074619293,
 0.6803277134895325]

In [12]:
def linguistic_acceptable_scores(texts):
    scores = []
    results = linguistic_acceptability(texts, function_to_apply='none', top_k=None)
    for result in results:
        for label in result:
            if label['label'] == 'LABEL_1':
                scores.append(label['score'])
    return scores

linguistic_acceptable_scores(["Her went to a school"])

[-1.2301501035690308]

In [13]:
text_test = ["Donald Trump is facing a widening crisis amid a report claiming that his name appears in US justice department files about Jeffrey Epstein as Congress subpoenas testimony from Epstein accomplice Ghislaine Maxwell."]
print(neutral_scores(text_test), linguistic_acceptable_scores(text_test))

[1.1469523906707764] [1.2529655694961548]


In [14]:
# Dataset preparation

In [15]:
dataset = load_dataset("argilla/news-summary")
dataset['train'][108]

{'text': '(Reuters) - Best known as a New York hedge fund industry executive, Anthony Scaramucci, President Donald Trump’s incoming communications director, has stakes in a film company, a glitzy Manhattan steakhouse and a nutrition business accused by U.S. regulators of making false claims in 2015, financial disclosures show. Overall, Scaramucci has assets in a range of approximately $61 million to $85 million, the forms show. He also has liabilities, such as mortgages and personal loans, of between $6.9 million and $25.8 million.  Scaramucci’s income since the start of 2016 - more than $10 million - is mostly derived from SkyBridge Capital, the hedge fund investment business that he founded in 2005 and is now in the process of selling to join the Trump administration. The disclosure says Scaramucci stands to make more than $50 million from the SkyBridge sale, which he said in May would likely close in June. The deal is on hold pending a regulatory review of its foreign-linked buyers.

In [16]:
flan_t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

In [17]:
dataset = dataset.remove_columns(
    ['prediction', 'prediction_agent', 'annotation', 'annotation_agent', 'metadata', 'status', 'event_timestamp', 'metrics']
)
dataset = dataset.map(
    lambda x: {"input_ids": flan_t5_tokenizer.encode('summarize: ' + x["text"], return_tensors="pt")},
    batched=False,
)
dataset.set_format("pytorch")
dataset['train'][108]

Map:   0%|          | 0/20417 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1500 > 512). Running this sequence through the model will result in indexing errors


{'text': '(Reuters) - Best known as a New York hedge fund industry executive, Anthony Scaramucci, President Donald Trump’s incoming communications director, has stakes in a film company, a glitzy Manhattan steakhouse and a nutrition business accused by U.S. regulators of making false claims in 2015, financial disclosures show. Overall, Scaramucci has assets in a range of approximately $61 million to $85 million, the forms show. He also has liabilities, such as mortgages and personal loans, of between $6.9 million and $25.8 million.  Scaramucci’s income since the start of 2016 - more than $10 million - is mostly derived from SkyBridge Capital, the hedge fund investment business that he founded in 2005 and is now in the process of selling to join the Trump administration. The disclosure says Scaramucci stands to make more than $50 million from the SkyBridge sale, which he said in May would likely close in June. The deal is on hold pending a regulatory review of its foreign-linked buyers.

In [18]:
# Flan T5

In [19]:
import wandb

ppo_config = PPOConfig(
    model_name="google/flan-t5-small",
    batch_size=16,
    mini_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    remove_unused_columns=False,
    log_with="wandb",
)

/home/huo/Workdir/venv/py3transformer/lib/python3.9/site-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(


In [20]:
flan_t5_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained("google/flan-t5-small")
flan_t5_model_ref = create_reference_model(flan_t5_model)

In [21]:
generation_kwargs = {
    "min_length": 64,
    "num_beams": 5,
    "no_repeat_ngram_size": 5,
    "do_sample": True,
    "pad_token_id": flan_t5_tokenizer.pad_token_id,
    "max_length": 256,
    "eos_token_id": flan_t5_tokenizer.eos_token_id,
}

In [22]:
def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

In [23]:
ppo_trainer = PPOTrainer(
    ppo_config, flan_t5_model, flan_t5_model_ref, flan_t5_tokenizer, dataset['train'], data_collator=collator
)

/home/huo/Workdir/venv/py3transformer/lib/python3.9/site-packages/trl/trainer/ppo_trainer.py:193: FutureWarning: `PPOTrainer` is deprecated and will be removed in trl v0.12. Please use `PPOv2Trainer` instead.
  warnings.warn(
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /home/huo/.netrc
wandb: Currently logged in as: firelouiszj (firelouiszj-opensee) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
